In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ===============================
# Load Dataset
# ===============================

data = pd.read_csv("pos_tags.csv")

print(data.head())

# ===============================
# Convert dataset into sentences
# ===============================

sentences = []
temp = []

for _, row in data.iterrows():

    word = row["word"]
    tag = row["tag"]

    temp.append((word, tag))

    if len(temp) == 20:
        sentences.append(temp)
        temp = []

print("Total Sentences:", len(sentences))

# ===============================
# Train Test Split
# ===============================

train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

# ===============================
# Vocabulary
# ===============================

vocabulary = set()
tags = set()

for sentence in train_data:

    for word, tag in sentence:

        vocabulary.add(word.lower())
        tags.add(tag)

vocabulary = list(vocabulary)
tags = list(tags)

word_to_index = {
    word: i
    for i, word in enumerate(vocabulary)
}

tag_to_index = {
    tag: i
    for i, tag in enumerate(tags)
}

index_to_tag = {
    i: tag
    for tag, i in tag_to_index.items()
}

V = len(vocabulary)
T = len(tags)

print("Vocabulary Size:", V)
print("Number of Tags:", T)

# ===============================
# HMM Matrices
# ===============================

initial = np.ones(T)

transition = np.ones((T, T))

emission = np.ones((T, V))

# ===============================
# Count Probabilities
# ===============================

for sentence in train_data:

    first_tag = sentence[0][1]

    initial[
        tag_to_index[first_tag]
    ] += 1

    for i, (word, tag) in enumerate(sentence):

        word = word.lower()

        tag_index = tag_to_index[tag]

        if word in word_to_index:

            emission[
                tag_index,
                word_to_index[word]
            ] += 1

        if i > 0:

            previous_tag = sentence[i - 1][1]

            transition[
                tag_to_index[previous_tag],
                tag_index
            ] += 1

# ===============================
# Normalize
# ===============================

initial /= initial.sum()

transition /= transition.sum(
    axis=1,
    keepdims=True
)

emission /= emission.sum(
    axis=1,
    keepdims=True
)

# ===============================
# Convert into Log Space
# ===============================

log_initial = np.log(initial)

log_transition = np.log(transition)

log_emission = np.log(emission)

# ===============================
# Vectorized Viterbi Algorithm
# ===============================

def viterbi(sentence):

    n = len(sentence)

    dp = np.zeros((T, n))

    backpointer = np.zeros((T, n), dtype=int)

    word = sentence[0].lower()

    if word in word_to_index:

        emit = log_emission[:, word_to_index[word]]

    else:

        emit = np.log(np.ones(T) * 1e-10)

    dp[:, 0] = log_initial + emit

    for i in range(1, n):

        word = sentence[i].lower()

        if word in word_to_index:

            emit = log_emission[
                :,
                word_to_index[word]
            ]

        else:

            emit = np.log(np.ones(T) * 1e-10)

        scores = (

            dp[:, i - 1][:, None]

            +

            log_transition

        )

        backpointer[:, i] = np.argmax(
            scores,
            axis=0
        )

        dp[:, i] = (

            np.max(
                scores,
                axis=0
            )

            +

            emit

        )

    best = np.argmax(dp[:, -1])

    result = [best]

    for i in range(n - 1, 0, -1):

        best = backpointer[
            best,
            i
        ]

        result.append(best)

    result.reverse()

    return [

        index_to_tag[i]

        for i in result

    ]

# ===============================
# Prediction on Test Set
# ===============================

actual = []

predicted = []

for sentence in test_data:

    words = [

        w

        for w, t in sentence

    ]

    true_tags = [

        t

        for w, t in sentence

    ]

    pred_tags = viterbi(words)

    actual.extend(true_tags)

    predicted.extend(pred_tags)

# ===============================
# Evaluation
# ===============================

print("\nAccuracy")

print(
    accuracy_score(
        actual,
        predicted
    )
)

print("\nClassification Report")

print(
    classification_report(
        actual,
        predicted,
        zero_division=0
    )
)

# ===============================
# Test on Five Unseen Sentences
# ===============================

test_sentences = [

    "Artificial intelligence improves healthcare systems".split(),

    "Students are learning natural language processing".split(),

    "The cat is sleeping peacefully".split(),

    "Python makes programming easier".split(),

    "They completed the assignment successfully".split()

]

for sentence in test_sentences:

    prediction = viterbi(sentence)

    print("\nSentence")

    print(" ".join(sentence))

    print("-------------------------")

    for word, tag in zip(sentence, prediction):

        print(word, "---->", tag)

   sentence_id    word  tag
0            0      aa   NN
1            1     aaa   NN
2            2     aah   NN
3            3   aahed  VBN
4            4  aahing  VBG
Total Sentences: 18505
Vocabulary Size: 296080
Number of Tags: 25

Accuracy
0.627681707646582

Classification Report
              precision    recall  f1-score   support

          CC       0.00      0.00      0.00         2
          CD       0.00      0.00      0.00         1
          DT       0.00      0.00      0.00         6
          IN       0.00      0.00      0.00        24
          JJ       0.00      0.00      0.00      7747
         JJR       0.00      0.00      0.00         3
         JJS       0.00      0.00      0.00        70
          MD       0.00      0.00      0.00         3
          NN       0.63      1.00      0.77     46461
         NNS       0.00      0.00      0.00      9689
         PRP       0.00      0.00      0.00         3
        PRP$       0.00      0.00      0.00         1
          RB